In [ ]:

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM  # Punkt 4 benötigt
import pickle
import nltk
from nltk.tokenize import PunktTokenizer
#nltk.download() # falls zu lange dauert, 
nltk.download('punkt_tab')
# Load from a file


In [ ]:
with open('../Dataset/10K_sample_over_all.pkl', 'rb') as file:
    df = pickle.load(file)

# ============================================
# 1. Chunking
# ============================================



In [ ]:
chunks = list()
metadaten = list()  # parallel zu chunks speichern
sentencer = PunktTokenizer()
#idx = 0
row_index = int(0)
for idx, row in df.iterrows():
    text = f"{row['abstract']}. {row['introduction']}"
    chunk_index = int(0)
    sentences = sentencer.tokenize(text)    
    for sentence in sentences:
        if sentence and len(sentence):            
            chunks.append(sentence)            
            # Metadaten zu diesem Chunk speichern
            metadaten.append((                 
                row_index,
                chunk_index  # within the row
            ))
            chunk_index = chunk_index + 1
    row_index = row_index + 1


In [ ]:

# ============================================
# 2. Embedding
# ============================================
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = embed_model.encode(chunks)


In [102]:
import sys
# ============================================
# 3. Retrieval mit Metadaten
# ============================================
def retrieve_with_metadata(query, k=2):
    query_emb = embed_model.encode([query])[0]
    similarities = np.dot(chunk_embeddings, query_emb) / (
        np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(query_emb)
    )
    top_k_idx = np.argsort(similarities)[-k:][::-1]
    
    # Chunks UND Metadaten zurückgeben
    result_chunks = [chunks[i] for i in top_k_idx]
    result_metadata = list()
    for i in top_k_idx:
        md = metadaten[i]
        row = df.iloc[md[0]]
        result_metadata.append({
                #'pmcid': row['pmcid'],
                'title': row['title'],
                #'year': row['year'],                
                #'chunk_index': md[1]  # vereinfacht
        })
    #print(result_metadata)
    return result_chunks, result_metadata


In [ ]:

# ============================================
# 4. LLM laden (hier FLAN-T5-small)
# ============================================
print("Lade LLM...")
llm_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(llm_name)


In [ ]:

# ============================================
# 5. Antwortgenerierung definieren
# ============================================
def generate_answer(query):
    # Relevante Chunks holen (Korrektur: Funktion heißt retrieve_with_metadata)
    chunks, metadata = retrieve_with_metadata(query, k=2)
    context = " "#.join(chunks)
    idx = 0
    
    for chunk in chunks:
          md = metadata[idx]
          idx = idx + 1
          chunk += f"titel: {md['title']}"
          context.join(chunk)

    # Prompt im Stil von FLAN-T5 (instruction-tuned)
    prompt = f"Answer the question based on the given context.\nContext: {context}\nQuestion: {query}\nAnswer:"

    # LLM aufrufen
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = llm.generate(
        **inputs, 
        max_new_tokens=100, 
        do_sample=True,        # WICHTIG: erst damit wird Temperatur aktiv
        temperature=0.3
    )
    antwort = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return antwort



In [130]:
frage = "What are risk factors for HPV?"
antwort = generate_answer(frage)
print("\nQuestion:", frage)
print("Answer:", antwort)


Question: What are risk factors for HPV?
Answer: high-risk and potentially carcinogenic types


In [131]:
frage = "What is bipolar disorder and szysophrenia?"
antwort = generate_answer(frage)
print("\nQuestion:", frage)
print("Answer:", antwort)


Question: What is bipolar disorder and szysophrenia?
Answer: a mood disorder marked by episodes of depression, (hypo)mania, and mixed episodes with interspersed subsyndromal symptoms in between episodes


In [89]:
frage = "What neurons comprise the leeches tactile system?"
antwort = generate_answer(frage)
print("\nQuestion:", frage)
print("Answer:", antwort)

[{'title': 'Decoding spatial location of perceived pain to acupuncture needle\nusing multivoxel pattern analysis The posterior parietal cortex remaps touch into external\nspace Distributed representation of single touches in somatosensory and\nvisual cortex Space coding in primate posterior parietal\ncortex The somatosensory cortex of human: cytoarchitecture and regional\ndistributions of receptor-binding sites Hierarchical somatosensory processing Distributed functions of detection and discrimination of\nvibrotactile stimuli in the hierarchical human somatosensory\nsystem Neural codes for perceptual discrimination in primary\nsomatosensory cortex Frontoparietal mechanisms supporting attention to location and\nintensity of painful stimuli Neural codes for somatosensory two-point discrimination in\ninferior parietal lobule: an fMRI study Somatosensation in social perception Distributed processing of pain and vibration by the human\nbrain A new view of pain as a homeostatic emotion Brain

In [68]:
frage = "What is the meaning of sanguivorous?"
antwort = generate_answer(frage)
print("\nQuestion:", frage)
print("Answer:", antwort)

type(md): <class 'tuple'>; md = (7991, 25), md[0] = 7991
type(md): <class 'tuple'>; md = (4043, 0), md[0] = 4043
chunk is Á Löve, Agropyron elongatum (Host) P. Beauvois.pcmcid: PMC11285282 titel: Development and identification of a novel wheat- Thinopyrum ponticum disomic substitution line DS5Ag(5D) with new genes conferring resistance to powdery mildew and leaf rust Cytogenetic identification and molecular marker development of a novel wheat- Thinopyrum ponticum translocation line with powdery mildew resistance Cloning of the broadly effective wheat leaf rust resistance gene Lr42 transferred from Aegilops tauschii Fighting wheat powdery mildew: from genes to fields Molecular cytogenetic identification of a novel wheat- Thinopyrum ponticum 1J S (1B) substitution line resistant to powdery mildew and leaf rust Physical mapping of a new powdery mildew resistance locus from Thinopyrum ponticum chromosome 4AgS Identification of wheat leaf rust resistance genes in chinese wheat cultivars and

metadata is <class 'list'>

Question: What is the meaning of sanguivorous?
Answer: a sacrificial sacrificial sacrificial


In [77]:
frage = "What is biology"
antwort = generate_answer(frage)
print("\nQuestion:", frage)
print("Answer:", antwort)


Question: What is biology
Answer: biology


type(md): <class 'tuple'>; md = (10351, 0), md[0] = 10351
type(md): <class 'tuple'>; md = (10351, 8), md[0] = 10351
chunk is Biochemistry and molecular biology is an experimental discipline and therefore training students in experimental techniques and data analysis is an essential component of undergraduate degrees.
chunk is Practical work has always been seen as an essential component of a biology degree.
